# Compare DNA ladders on an illustrative gel

Load two Penn State markers, switch between them on the same samples and export an SVG with its lane inventory. The model places linear DNA on a logarithmic size scale. Untreated circular pUC19 stays in the inventory because its migration depends on conformation. [Ladder sources and attribution](../../databases/ladders/README.md)

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import sys

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "strandfolio" / "__init__.py").is_file())
sys.path.insert(0, str(root))

from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from IPython.display import SVG, display

from strandfolio.domain.sequence import SequenceRevision
from strandfolio.io.formats import readSequence
from strandfolio.operations.gel import GelMarker, GelState, untreatedLane
from strandfolio.rendering.gels import buildGelLayout, gelData, gelSvg
from strandfolio.storage.catalogues import DEFAULT_LADDER_ID, loadDnaLadders

## Choose two size markers

`loadDnaLadders()` reads the distributed catalogue. Each profile carries the exact component sizes in bp, release identity and attribution.

In [ ]:
profiles = {profile.ladderId: profile for profile in loadDnaLadders()}
extended = GelMarker.fromProfile(profiles[DEFAULT_LADDER_ID])
short = GelMarker.fromProfile(profiles["psu-100bp"])

for marker in (extended, short):
    low, high = marker.rangeBp
    print(f"{marker.profile.name}: {low:,}–{high:,} bp")

## Add linear standards and the circular plasmid

Use synthetic linear DNA at three known lengths to see how the model handles its 100–10,000 bp display range. Read the annotated pUC19 GenBank file for the untreated plasmid lane.

In [ ]:
def linearLane(lengthBp):
    name = f"{lengthBp:,} bp standard"
    record = SeqRecord(Seq("A" * lengthBp), id=name, name=name,
                       annotations={"molecule_type": "ds-DNA", "topology": "linear"})
    return untreatedLane(SequenceRevision.fromSeqRecord(record, moleculeType="DNA"))


plasmid = readSequence(root / "databases/plasmids/maps/pUC19.gb").records[0]
gel = GelState(extended, lanes=(linearLane(1000), linearLane(200), linearLane(50),
                               untreatedLane(plasmid)))

## Switch to the 100 bp marker

The 200 bp standard falls below the extended marker's range and inside the 100 bp marker's range. Switching the marker keeps the same sample lanes and display scale. The 50 bp standard stays below the display range. Equal-sized marker components share a band.

In [ ]:
gel = gel.withMarker(short)
for band in buildGelLayout(gel).marker.bands:
    if len(band.markerComponents) > 1:
        print(f"{band.sizeBp:,} bp band: {len(band.markerComponents)} marker components")

figure = gelSvg(gel)
display(SVG(data=figure))

## Save the figure and lane inventory

The SVG includes the marker attribution and scale. The JSON inventory retains every sample and its display status, including the off-scale standard and circular pUC19.

In [ ]:
runId = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
outputDir = root / "docs/notebooks/_outputs/L01" / runId
outputDir.mkdir(parents=True)
(outputDir / "synthetic-gel.svg").write_text(figure, encoding="utf-8")
(outputDir / "inventory.json").write_text(json.dumps(gelData(gel), indent=2) + "\n", encoding="utf-8")
print(outputDir.relative_to(root))